In [0]:
#Connection Of Storage Account
storage_account_name = "storage12accountgp"
storage_account_key = "yQhF8dKEAkDS2qyacCpz7/xYGnF8UR9VhxSSGgBsbE+YBJlQ4ixBUWozg4elJ5QWIOi4rmMprja3+AStsm80uw=="

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.blob.core.windows.net",
    storage_account_key
)

print("Storage connected successfully!")

Storage connected successfully!


In [0]:
#Bronze Data (day-wise)
bronze_path = f"wasbs://bronze-layer@{storage_account_name}.blob.core.windows.net/"
files = dbutils.fs.ls(bronze_path)
for f in files:
    print(f.path)

wasbs://bronze-layer@storage12accountgp.blob.core.windows.net/day 1/
wasbs://bronze-layer@storage12accountgp.blob.core.windows.net/day 2/
wasbs://bronze-layer@storage12accountgp.blob.core.windows.net/day 3/
wasbs://bronze-layer@storage12accountgp.blob.core.windows.net/day 4/
wasbs://bronze-layer@storage12accountgp.blob.core.windows.net/day 5/


In [0]:
# Reading Day 1 Data
day1_path = f"wasbs://bronze-layer@{storage_account_name}.blob.core.windows.net/day 1/"

df_day1 = spark.read.csv(day1_path, header=True, inferSchema=True)

# Number of Rows and Schema
df_day1.printSchema()
df_day1.show(5)
print(f"Total rows: {df_day1.count()}")

root
 |-- emp_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- salary: integer (nullable = true)
 |-- join_date: date (nullable = true)
 |-- status: string (nullable = true)

+------+-------+----------+------+----------+------+
|emp_id|   name|department|salary| join_date|status|
+------+-------+----------+------+----------+------+
|   151|Emp_151| Marketing| 73963|2020-03-08|Active|
|   152|Emp_152| Marketing| 79253|2020-03-31|Active|
|   153|Emp_153|        IT| 83668|2021-10-16|Active|
|   154|Emp_154|   Finance| 84331|2020-06-28|Active|
|   155|Emp_155| Marketing| 90570|2021-12-28|Active|
+------+-------+----------+------+----------+------+
only showing top 5 rows

Total rows: 200


In [0]:
#Reading all days data with Day/Date Tracking + SCHEMA DRIFT HANDLING
from pyspark.sql.functions import lit, to_date
days_info = [
    ("day 1", "2026-07-07"),
    ("day 2", "2026-07-08"),
    ("day 3", "2026-07-09"),
    ("day 4", "2026-07-10"),
    ("day 5", "2026-07-11"),
]

all_days_df = None
baseline_columns = None

for folder_name, date_str in days_info:
    path = f"wasbs://bronze-layer@{storage_account_name}.blob.core.windows.net/{folder_name}/"
    
    df = spark.read.csv(path, header=True, inferSchema=True)

    # --- Schema drift check: compare this day's columns against Day 1's baseline ---
    current_columns = set(df.columns)
    if baseline_columns is None:
        baseline_columns = current_columns
        print(f"{folder_name}: baseline schema set -> {sorted(baseline_columns)}")
    else:
        new_cols = current_columns - baseline_columns
        missing_cols = baseline_columns - current_columns
        if new_cols:
            print(f"{folder_name}: NEW column(s) detected -> {sorted(new_cols)}")
        if missing_cols:
            print(f"{folder_name}: MISSING column(s) vs baseline -> {sorted(missing_cols)}")
        if not new_cols and not missing_cols:
            print(f"{folder_name}: schema matches baseline, no drift")

    df = df.withColumn("batch_day", lit(folder_name)) \
           .withColumn("batch_date", to_date(lit(date_str)))
    
    if all_days_df is None:
        all_days_df = df
    else:
        all_days_df = all_days_df.unionByName(df, allowMissingColumns=True)

#Total rows across all days
all_days_df.show(10)
print(f"Total rows across all days: {all_days_df.count()}")

day 1: baseline schema set -> ['department', 'emp_id', 'join_date', 'name', 'salary', 'status']
day 2: schema matches baseline, no drift
day 3: schema matches baseline, no drift
day 4: schema matches baseline, no drift
day 5: schema matches baseline, no drift
+------+-------+----------+------+----------+------+---------+----------+
|emp_id|   name|department|salary| join_date|status|batch_day|batch_date|
+------+-------+----------+------+----------+------+---------+----------+
|   151|Emp_151| Marketing| 73963|2020-03-08|Active|    day 1|2026-07-07|
|   152|Emp_152| Marketing| 79253|2020-03-31|Active|    day 1|2026-07-07|
|   153|Emp_153|        IT| 83668|2021-10-16|Active|    day 1|2026-07-07|
|   154|Emp_154|   Finance| 84331|2020-06-28|Active|    day 1|2026-07-07|
|   155|Emp_155| Marketing| 90570|2021-12-28|Active|    day 1|2026-07-07|
|   156|Emp_156| Marketing| 47947|2020-07-23|Active|    day 1|2026-07-07|
|   157|Emp_157|        HR| 80039|2020-10-23|Active|    day 1|2026-07-07|


In [0]:
from pyspark.sql.functions import lit, col

# Extracting day 1 data (starting point)
day1_df = all_days_df.filter(col("batch_day") == "day 1")

# Convert into SCD Type 2 
silver_history_init = day1_df.select(
    "emp_id", "name", "department", "salary", "join_date", "status"
).withColumn("start_date", to_date(lit("2026-07-07"))) \
 .withColumn("end_date", lit("9999-12-31").cast("date")) \
 .withColumn("is_current", lit(1))

# Create Delta Table and Save it 
silver_path = f"wasbs://bronze-layer@{storage_account_name}.blob.core.windows.net/../silver-layer/employee_history"

silver_history_init.write.format("delta").mode("overwrite").save(
    f"wasbs://silver-layer@{storage_account_name}.blob.core.windows.net/employee_history"
)

print("Silver history table initialized with Day 1 data!")
silver_history_init.show(5)

Silver history table initialized with Day 1 data!
+------+-------+----------+------+----------+------+----------+----------+----------+
|emp_id|   name|department|salary| join_date|status|start_date|  end_date|is_current|
+------+-------+----------+------+----------+------+----------+----------+----------+
|   151|Emp_151| Marketing| 73963|2020-03-08|Active|2026-07-07|9999-12-31|         1|
|   152|Emp_152| Marketing| 79253|2020-03-31|Active|2026-07-07|9999-12-31|         1|
|   153|Emp_153|        IT| 83668|2021-10-16|Active|2026-07-07|9999-12-31|         1|
|   154|Emp_154|   Finance| 84331|2020-06-28|Active|2026-07-07|9999-12-31|         1|
|   155|Emp_155| Marketing| 90570|2021-12-28|Active|2026-07-07|9999-12-31|         1|
+------+-------+----------+------+----------+------+----------+----------+----------+
only showing top 5 rows



In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import col, lit, to_date

silver_path = f"wasbs://silver-layer@{storage_account_name}.blob.core.windows.net/employee_history"

# Creating Delta Table Object for Merge Operation 
delta_table = DeltaTable.forPath(spark, silver_path)

print("Delta table loaded, ready for MERGE operations!")

Delta table loaded, ready for MERGE operations!


In [0]:
#Merge Loop from day 2 to day 5
days_to_process = [
    ("day 2", "2026-07-08"),
    ("day 3", "2026-07-09"),
    ("day 4", "2026-07-10"),
    ("day 5", "2026-07-11"),
]

for folder_name, date_str in days_to_process:
    print(f"\n--- Processing {folder_name} ---")

    new_day_df = all_days_df.filter(col("batch_day") == folder_name) \
        .select("emp_id", "name", "department", "salary", "join_date", "status") \
        .withColumn("new_start_date", to_date(lit(date_str)))
    incoming_count = new_day_df.count()
    print(f"{folder_name}: {incoming_count} incoming source rows")

    #Step 1 Close old current records where there is a data change
    merge_result = delta_table.alias("target").merge(
        new_day_df.alias("source"),
        "target.emp_id = source.emp_id AND target.is_current = 1"
    ).whenMatchedUpdate(
        condition="""
            target.department != source.department OR
            target.salary != source.salary OR
            target.status != source.status
        """,
        set={
            "end_date": "source.new_start_date",
            "is_current": lit(0)
        }
    ).execute()

    #Step 2 Insert new records for new employees / changed employees
    current_records = delta_table.toDF().filter(col("is_current") == 1)

    changed_or_new = new_day_df.alias("new").join(
        current_records.alias("cur"),
        on="emp_id",
        how="left"
    ).filter(
        col("cur.emp_id").isNull() |  # naya employee
        (col("new.department") != col("cur.department")) |
        (col("new.salary") != col("cur.salary")) |
        (col("new.status") != col("cur.status"))
    ).select(
        "new.emp_id", "new.name", "new.department", "new.salary",
        "new.join_date", "new.status", "new.new_start_date"
    )

    to_insert = changed_or_new.withColumn("start_date", col("new_start_date")) \
        .withColumn("end_date", lit("9999-12-31").cast("date")) \
        .withColumn("is_current", lit(1)) \
        .select("emp_id", "name", "department", "salary", "join_date", "status", "start_date", "end_date", "is_current") \
        .cache()

    inserted_count = to_insert.count()

    if inserted_count > 0:
        to_insert.write.format("delta").mode("append").save(silver_path)

    print(f"{folder_name}: {inserted_count} new/changed records inserted (out of {incoming_count} incoming rows)")
    if inserted_count == 0:
        print(f"  -> WARNING: 0 changes detected for {folder_name}. If this is a fresh run, "
              f"double-check the join/compare logic above instead of assuming it's correct.")

print("\n SCD Type 2 processing complete for all days!")


--- Processing day 2 ---
day 2: 205 incoming source rows
day 2: 83 new/changed records inserted (out of 205 incoming rows)

--- Processing day 3 ---
day 3: 210 incoming source rows
day 3: 103 new/changed records inserted (out of 210 incoming rows)

--- Processing day 4 ---
day 4: 215 incoming source rows
day 4: 101 new/changed records inserted (out of 215 incoming rows)

--- Processing day 5 ---
day 5: 220 incoming source rows
day 5: 107 new/changed records inserted (out of 220 incoming rows)

 SCD Type 2 processing complete for all days!


In [0]:
# Complete history table 
history_df = spark.read.format("delta").load(silver_path)

print(f"Total history records: {history_df.count()}")

# Specific History record according to employee id
history_df.filter(col("emp_id") == 1).orderBy("start_date").show(truncate=False)

# Current records 
print(f"Current active records: {history_df.filter(col('is_current') == 1).count()}")

Total history records: 594
+------+-----+----------+------+----------+------+----------+----------+----------+
|emp_id|name |department|salary|join_date |status|start_date|end_date  |is_current|
+------+-----+----------+------+----------+------+----------+----------+----------+
|1     |Emp_1|Sales     |46632 |2020-07-30|Active|2026-07-07|2026-07-08|0         |
|1     |Emp_1|Marketing |46632 |2020-07-30|Active|2026-07-08|2026-07-09|0         |
|1     |Emp_1|Finance   |46632 |2020-07-30|Active|2026-07-09|2026-07-11|0         |
|1     |Emp_1|Finance   |54780 |2020-07-30|Active|2026-07-11|9999-12-31|1         |
+------+-----+----------+------+----------+------+----------+----------+----------+

Current active records: 220


In [0]:
from pyspark.sql.functions import col, current_date, datediff, round as spark_round

# current records 
current_snapshot = history_df.filter(col("is_current") == 1)

# Tenure calculation
current_snapshot = current_snapshot.withColumn(
    "tenure_years",
    spark_round(datediff(current_date(), col("join_date")) / 365.25, 2)
)

# current_version_start_date is also there
current_snapshot = current_snapshot.withColumnRenamed("start_date", "current_version_start_date")

# Selection of Final columns 
current_snapshot_final = current_snapshot.select(
    "emp_id", "name", "department", "salary", "join_date",
    "status", "current_version_start_date", "tenure_years"
)

# Saving Delta table 
current_snapshot_final.write.format("delta").mode("overwrite").save(
    f"wasbs://silver-layer@{storage_account_name}.blob.core.windows.net/employee_current_snapshot"
)

print(f"Current snapshot created! Total rows: {current_snapshot_final.count()}")
current_snapshot_final.show(5)

Current snapshot created! Total rows: 220
+------+------+----------+------+----------+------+--------------------------+------------+
|emp_id|  name|department|salary| join_date|status|current_version_start_date|tenure_years|
+------+------+----------+------+----------+------+--------------------------+------------+
|    19|Emp_19|        IT| 36875|2022-08-05|Active|                2026-07-07|        3.93|
|    29|Emp_29|        HR| 39637|2021-04-27|Active|                2026-07-07|         5.2|
|    31|Emp_31|     Sales| 48997|2020-08-09|Active|                2026-07-07|        5.92|
|    33|Emp_33|     Sales| 78953|2020-05-19|Active|                2026-07-07|        6.14|
|    36|Emp_36|   Finance| 46673|2020-02-02|Active|                2026-07-07|        6.44|
+------+------+----------+------+----------+------+--------------------------+------------+
only showing top 5 rows



In [0]:
# Converting Silver tables to temp views  to run SQL  query 
history_df.createOrReplaceTempView("employee_history")
current_snapshot_final.createOrReplaceTempView("employee_current_snapshot")

print("Temp views created! Now we can run SQL Queries.")

Temp views created! Now we can run SQL Queries.


In [0]:
# WINDOW FUNCTION EXAMPLE: track each employee's previous department/salary
# using LAG, so we can see exactly what changed at every SCD2 version boundary.
salary_and_dept_changes = spark.sql("""
    SELECT
        emp_id,
        name,
        start_date,
        department,
        salary,
        status,
        LAG(department) OVER (PARTITION BY emp_id ORDER BY start_date) AS prev_department,
        LAG(salary)     OVER (PARTITION BY emp_id ORDER BY start_date) AS prev_salary,
        salary - LAG(salary) OVER (PARTITION BY emp_id ORDER BY start_date) AS salary_change,
        ROW_NUMBER() OVER (PARTITION BY emp_id ORDER BY start_date) AS version_number
    FROM employee_history
    ORDER BY emp_id, start_date
""")

print("Employees whose department changed at least once:")
salary_and_dept_changes.filter(
    (col("prev_department").isNotNull()) & (col("department") != col("prev_department"))
).show(10)

print("Employees who got a raise (salary_change > 0):")
salary_and_dept_changes.filter(col("salary_change") > 0).show(10)

Employees whose department changed at least once:
+------+------+----------+----------+------+------+---------------+-----------+-------------+--------------+
|emp_id|  name|start_date|department|salary|status|prev_department|prev_salary|salary_change|version_number|
+------+------+----------+----------+------+------+---------------+-----------+-------------+--------------+
|     1| Emp_1|2026-07-08| Marketing| 46632|Active|          Sales|      46632|            0|             2|
|     1| Emp_1|2026-07-09|   Finance| 46632|Active|      Marketing|      46632|            0|             3|
|     2| Emp_2|2026-07-08|   Finance| 65513|Active|             HR|      65513|            0|             2|
|     2| Emp_2|2026-07-10|     Sales| 65513|Active|        Finance|      65513|            0|             3|
|     3| Emp_3|2026-07-10|        IT| 93010|Active|          Sales|      93010|            0|             3|
|     7| Emp_7|2026-07-09|        IT| 30599|Active|          Sales|      30599

In [0]:
headcount_by_department = spark.sql("""
    SELECT 
        batch_day,
        batch_date AS date,
        department,
        COUNT(*) AS headcount
    FROM (
        SELECT 
            h.emp_id,
            h.department,
            d.batch_day,
            d.batch_date
        FROM employee_history h
        JOIN (
            SELECT DISTINCT batch_day, batch_date FROM 
            (SELECT 'day 1' AS batch_day, DATE('2026-07-07') AS batch_date
             UNION ALL SELECT 'day 2', DATE('2026-07-08')
             UNION ALL SELECT 'day 3', DATE('2026-07-09')
             UNION ALL SELECT 'day 4', DATE('2026-07-10')
             UNION ALL SELECT 'day 5', DATE('2026-07-11'))
        ) d
        ON h.start_date <= d.batch_date 
           AND (h.end_date > d.batch_date)
           AND h.status = 'Active'
    )
    GROUP BY batch_day, date, department
    ORDER BY batch_day, department
""")

headcount_by_department.show(30)

+---------+----------+----------+---------+
|batch_day|      date|department|headcount|
+---------+----------+----------+---------+
|    day 1|2026-07-07|   Finance|       50|
|    day 1|2026-07-07|        HR|       41|
|    day 1|2026-07-07|        IT|       38|
|    day 1|2026-07-07| Marketing|       37|
|    day 1|2026-07-07|     Sales|       34|
|    day 2|2026-07-08|   Finance|       47|
|    day 2|2026-07-08|        HR|       48|
|    day 2|2026-07-08|        IT|       35|
|    day 2|2026-07-08| Marketing|       34|
|    day 2|2026-07-08|     Sales|       41|
|    day 3|2026-07-09|   Finance|       44|
|    day 3|2026-07-09|        HR|       48|
|    day 3|2026-07-09|        IT|       35|
|    day 3|2026-07-09| Marketing|       36|
|    day 3|2026-07-09|     Sales|       38|
|    day 4|2026-07-10|   Finance|       41|
|    day 4|2026-07-10|        HR|       40|
|    day 4|2026-07-10|        IT|       41|
|    day 4|2026-07-10| Marketing|       33|
|    day 4|2026-07-10|     Sales

In [0]:
attrition_summary = spark.sql("""
    WITH day_dates AS (
        SELECT 'day 1' AS batch_day, DATE('2026-07-07') AS batch_date
        UNION ALL SELECT 'day 2', DATE('2026-07-08')
        UNION ALL SELECT 'day 3', DATE('2026-07-09')
        UNION ALL SELECT 'day 4', DATE('2026-07-10')
        UNION ALL SELECT 'day 5', DATE('2026-07-11')
    ),
    active_hc AS (
        SELECT 
            d.batch_day,
            d.batch_date,
            h.department,
            COUNT(DISTINCT h.emp_id) AS active_count
        FROM employee_history h
        JOIN day_dates d
            ON h.start_date <= d.batch_date 
            AND h.end_date > d.batch_date
            AND h.status = 'Active'
        GROUP BY d.batch_day, d.batch_date, h.department
    ),
    resign_hc AS (
        SELECT 
            d.batch_day,
            d.batch_date,
            h.department,
            COUNT(DISTINCT h.emp_id) AS resign_count
        FROM employee_history h
        JOIN day_dates d
            ON h.start_date = d.batch_date 
            AND h.status = 'Resigned'
        GROUP BY d.batch_day, d.batch_date, h.department
    ),
    all_depts AS (
        SELECT DISTINCT d.batch_day, d.batch_date, dept.department
        FROM day_dates d
        CROSS JOIN (SELECT DISTINCT department FROM employee_history) dept
    ),
    combined AS (
        SELECT 
            ad.batch_day,
            ad.batch_date AS date,
            ad.department,
            COALESCE(rh.resign_count, 0) AS resignations,
            COALESCE(ah.active_count, 0) AS active_count,
            (COALESCE(ah.active_count, 0) + COALESCE(rh.resign_count, 0)) AS headcount_at_risk
        FROM all_depts ad
        LEFT JOIN active_hc ah 
            ON ad.batch_day = ah.batch_day AND ad.department = ah.department
        LEFT JOIN resign_hc rh 
            ON ad.batch_day = rh.batch_day AND ad.department = rh.department
    )
    SELECT 
        batch_day,
        date,
        department,
        resignations,
        headcount_at_risk,
        ROUND(
            CASE WHEN headcount_at_risk > 0 
                 THEN (CAST(resignations AS DOUBLE) / headcount_at_risk) * 100 
                 ELSE 0.0 
            END, 
            2
        ) AS attrition_rate_pct
    FROM combined
    ORDER BY batch_day, department
""")

attrition_summary.show(30)

+---------+----------+----------+------------+-----------------+------------------+
|batch_day|      date|department|resignations|headcount_at_risk|attrition_rate_pct|
+---------+----------+----------+------------+-----------------+------------------+
|    day 1|2026-07-07|   Finance|           0|               50|               0.0|
|    day 1|2026-07-07|        HR|           0|               41|               0.0|
|    day 1|2026-07-07|        IT|           0|               38|               0.0|
|    day 1|2026-07-07| Marketing|           0|               37|               0.0|
|    day 1|2026-07-07|     Sales|           0|               34|               0.0|
|    day 2|2026-07-08|   Finance|           0|               47|               0.0|
|    day 2|2026-07-08|        HR|           0|               48|               0.0|
|    day 2|2026-07-08|        IT|           0|               35|               0.0|
|    day 2|2026-07-08| Marketing|           0|               34|            

In [0]:
tenure_summary = spark.sql("""
    SELECT 
        department,
        COUNT(*) AS employee_count,
        ROUND(AVG(tenure_years), 2) AS avg_tenure_years
    FROM employee_current_snapshot
    WHERE status = 'Active'
    GROUP BY department
    ORDER BY department
""")

tenure_summary.show()

+----------+--------------+----------------+
|department|employee_count|avg_tenure_years|
+----------+--------------+----------------+
|   Finance|            36|            4.56|
|        HR|            45|            4.54|
|        IT|            42|             4.4|
| Marketing|            26|            5.14|
|     Sales|            47|            5.14|
+----------+--------------+----------------+



In [0]:
# ---------- Saving SILVER LAYER tables to CSV format (for dashboard ) ----------

# 1. employee_history_log
history_df.write.format("csv").mode("overwrite").option("header", "true").save(
    f"wasbs://gold-layer@{storage_account_name}.blob.core.windows.net/employee_history_log"
)

# 2. employee_current_snapshot
current_snapshot_final.write.format("csv").mode("overwrite").option("header", "true").save(
    f"wasbs://gold-layer@{storage_account_name}.blob.core.windows.net/employee_current_snapshot"
)

# ---------- GOLD LAYER tables ----------

# 3. headcount_by_department
headcount_by_department.write.format("csv").mode("overwrite").option("header", "true").save(
    f"wasbs://gold-layer@{storage_account_name}.blob.core.windows.net/headcount_by_department"
)

# 4. attrition_summary
attrition_summary.write.format("csv").mode("overwrite").option("header", "true").save(
    f"wasbs://gold-layer@{storage_account_name}.blob.core.windows.net/attrition_summary"
)

# 5. tenure_summary
tenure_summary.write.format("csv").mode("overwrite").option("header", "true").save(
    f"wasbs://gold-layer@{storage_account_name}.blob.core.windows.net/tenure_summary"
)

print("All 5 output tables saved to gold-layer container!")

All 5 output tables saved to gold-layer container!


In [0]:
print("=== SANITY CHECK SUMMARY ===\n")

print(f"1. Total history records: {history_df.count()}  (Reference: 594)")
print(f"2. Current snapshot records: {current_snapshot_final.count()}  (Reference: 220)")
print(f"3. Total active employees (day 5): {current_snapshot_final.filter(col('status')=='Active').count()}")
print(f"4. Total resigned employees (day 5): {current_snapshot_final.filter(col('status')=='Resigned').count()}")

print("\n=== Tenure Summary (compare with tenure_summary.csv) ===")
tenure_summary.show()

print("\n=== Department count check ===")
tenure_summary.select("department").distinct().show()

=== SANITY CHECK SUMMARY ===

1. Total history records: 594  (Reference: 594)
2. Current snapshot records: 220  (Reference: 220)
3. Total active employees (day 5): 196
4. Total resigned employees (day 5): 24

=== Tenure Summary (compare with tenure_summary.csv) ===
+----------+--------------+----------------+
|department|employee_count|avg_tenure_years|
+----------+--------------+----------------+
|   Finance|            36|            4.56|
|        HR|            45|            4.54|
|        IT|            42|             4.4|
| Marketing|            26|            5.14|
|     Sales|            47|            5.14|
+----------+--------------+----------------+


=== Department count check ===
+----------+
|department|
+----------+
|   Finance|
|        HR|
|        IT|
| Marketing|
|     Sales|
+----------+

